In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score,average_precision_score,classification_report, confusion_matrix, recall_score
import optuna


/home/emivazcru/miniconda3/envs/evc1_314/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
col_selection = ['case', 'nhc_final', 'start_frame', 'end_frame', 'organ', 'organ_num', 
                 'any_prolapse', 'cystocele', 'cystourethrocele', 'uterine_prolapse', 
                 'cervical_elongation', 'rectocele', 'enterocele',
                 '1.1_frame_conf_mean_mean',
                 '1.1_frame_conf_mean_std', '1.1_frame_conf_mean_max',
                 '1.1_frame_conf_mean_min', '1.2_frame_conf_std_mean',
                 '1.2_frame_conf_std_std', '1.2_frame_conf_std_max',
                 '1.2_frame_conf_std_min', '1.3_frame_conf_max_mean',
                 '1.3_frame_conf_max_std', '1.3_frame_conf_max_max',
                 '1.3_frame_conf_max_min', '1.4_frame_conf_min_mean',
                 '1.4_frame_conf_min_std', '1.4_frame_conf_min_max',
                 '1.4_frame_conf_min_min', '2.1_region_conf_mean_mean',
                 '2.1_region_conf_mean_std', '2.1_region_conf_mean_max',
                 '2.1_region_conf_mean_min', '2.2_region_conf_std_mean',
                 '2.2_region_conf_std_std', '2.2_region_conf_std_max',
                 '2.2_region_conf_std_min', '2.3_region_conf_max_mean',
                 '2.3_region_conf_max_std', '2.3_region_conf_max_max',
                 '2.3_region_conf_max_min', '2.4_region_conf_min_mean',
                 '2.4_region_conf_min_std', '2.4_region_conf_min_max',
                 '2.4_region_conf_min_min', '3.1_region_coords_centroid_Y_mean',
                 '3.1_region_coords_centroid_Y_std', '3.1_region_coords_centroid_Y_max',
                 '3.1_region_coords_centroid_Y_min', '3.2_region_coords_centroid_X_mean',
                 '3.2_region_coords_centroid_X_std', '3.2_region_coords_centroid_X_max',
                 '3.2_region_coords_centroid_X_min', '3.3_region_coords_max_Y_mean',
                 '3.3_region_coords_max_Y_std', '3.3_region_coords_max_Y_max',
                 '3.3_region_coords_max_Y_min', '3.4_region_coords_max_X_mean',
                 '3.4_region_coords_max_X_std', '3.4_region_coords_max_X_max',
                 '3.4_region_coords_max_X_min', '3.5_region_coords_min_Y_mean',
                 '3.5_region_coords_min_Y_std', '3.5_region_coords_min_Y_max',
                 '3.5_region_coords_min_Y_min', '3.6_region_coords_min_X_mean',
                 '3.6_region_coords_min_X_std', '3.6_region_coords_min_X_max',
                 '3.6_region_coords_min_X_min', '3.7_region_coords_len_Y_mean',
                 '3.7_region_coords_len_Y_std', '3.7_region_coords_len_Y_max',
                 '3.7_region_coords_len_Y_min', '3.8_region_coords_len_X_mean',
                 '3.8_region_coords_len_X_std', '3.8_region_coords_len_X_max',
                 '3.8_region_coords_len_X_min', '3.9_region_coords_bbox_area_mean',
                 '3.9_region_coords_bbox_area_std', '3.9_region_coords_bbox_area_max',
                 '3.9_region_coords_bbox_area_min']

prolapses = ['any_prolapse', 'cystocele', 'cystourethrocele', 'uterine_prolapse', 
                 'cervical_elongation', 'rectocele', 'enterocele']

DF_PATH = 'data/case_level_feats_alltargets_v2.csv'

In [8]:
def load_data(prolapse:int = 0):
    ignored_obj = np.delete(prolapses, [prolapse])
    df = pd.read_csv(DF_PATH, usecols=lambda col: col not in ignored_obj)
    data,objective = df.drop(prolapses[prolapse],axis=1),df[prolapses[prolapse]]
    
    data['organ'] = data['organ'].astype('category')
    
    return data,objective
d,o = load_data()
d

,case,nhc_final,start_frame,end_frame,organ,organ_num,1.1_frame_conf_mean_mean,1.1_frame_conf_mean_std,1.1_frame_conf_mean_max,1.1_frame_conf_mean_min,...,3.7_region_coords_len_Y_max,3.7_region_coords_len_Y_min,3.8_region_coords_len_X_mean,3.8_region_coords_len_X_std,3.8_region_coords_len_X_max,3.8_region_coords_len_X_min,3.9_region_coords_bbox_area_mean,3.9_region_coords_bbox_area_std,3.9_region_coords_bbox_area_max,3.9_region_coords_bbox_area_min
0,28,28963,0,60,Anus,0.0,0.013236,0.003297,0.018074,0.006334,...,29.0,24.0,17.016667,1.408108,19.0,14.0,440.566667,48.237185,532.0,336.0
1,28,28963,0,60,Bladder,1.0,0.053493,0.005537,0.064626,0.039998,...,44.0,29.0,38.300000,1.843909,42.0,35.0,1399.916667,175.727571,1848.0,1044.0
2,28,28963,0,60,Levator ani muscle,2.0,0.006237,0.001692,0.010622,0.003167,...,34.0,7.0,10.466667,3.605395,24.0,6.0,142.450000,146.429879,792.0,42.0
3,28,28963,0,60,Pubis,3.0,0.012892,0.001698,0.016616,0.009995,...,21.0,16.0,17.766667,10.136972,45.0,11.0,341.383333,196.956856,900.0,192.0
4,28,28963,0,60,Rectum,4.0,0.058127,0.007759,0.074797,0.047989,...,55.0,41.0,34.583333,1.521503,38.0,32.0,1624.250000,200.061061,2090.0,1376.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5411,137,80570419,60,120,Pubis,3.0,0.016008,0.001040,0.017545,0.013704,...,17.0,14.0,19.550000,0.964189,21.0,17.0,309.916667,23.322566,357.0,266.0
5412,137,80570419,60,120,Rectum,4.0,0.060405,0.002993,0.067472,0.055078,...,51.0,43.0,30.116667,1.009978,32.0,28.0,1397.500000,83.073074,1568.0,1232.0
5413,137,80570419,60,120,Urethra,5.0,0.011541,0.000357,0.012312,0.010852,...,30.0,27.0,9.283333,0.613179,10.0,8.0,265.683333,19.244627,300.0,224.0
5414,137,80570419,60,120,Uterus,6.0,0.064225,0.002598,0.071763,0.059000,...,37.0,28.0,42.866667,1.346265,46.0,41.0,1324.750000,82.334914,1554.0,1148.0


In [9]:
def optimal_params(trial: optuna.Trial, data = None, objective = None):
    if data is None or objective is None:
        data,objective = load_data()
    
    sgkf = StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=66)
    groups = data["case"]
    data = data.drop(columns=["case"])
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'random_state': 66,
        'enable_categorical':True
    }
    
    scores = []
    
    for i, (train_index, test_index) in enumerate(sgkf.split(data, objective, groups)):
        #print(f"Fold {i}:")
        #print(f"  Train: index={train_index}")
        #print(f"         group={np.unique(groups[train_index])}")
        #print(f"  Test:  index={test_index}")
        #print(f"         group={np.unique(groups[test_index])}")
        assert set(np.unique(groups[train_index])).isdisjoint(set(np.unique(groups[test_index])))
        
        train, train_obj = data.iloc[train_index],objective.iloc[train_index]
        test, test_obj = data.iloc[test_index], objective.iloc[test_index]
        
        pos_weight = len(train_obj[train_obj==False]) / len(train_obj[train_obj==True])

        xgbc = XGBClassifier(**params, scale_pos_weight=pos_weight)
        xgbc.fit(X=train,y=train_obj)
        
        prediction = xgbc.predict_proba(X=test)[:,1]
        #prediction = xgbc.predict(X=test)
        scores.append(average_precision_score(y_true=test_obj,y_score=prediction))
        #scores.append(recall_score(y_true=test_obj,y_pred=prediction))
    return np.mean(scores)

In [13]:
class Results:
    def __init__(self, classif_report, conf_matrix):
        self.classif_report = classif_report
        self.conf_matrix = conf_matrix
        
    def __str__(self):
        
        return f"Report:\n{self.classif_report}\nMatrix:\n{self.conf_matrix}"
    def __repr__(self):
        return self.__str__()

def main_func():
    res = dict()
    
    for p in range(0,1): #len(prolapses)-1
        data,objective = load_data(p)
        groups = data["case"] #Eliminar esta columna en el entreno?
        sgkf = StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=66)
        train_index, eval_index = next(sgkf.split(data, objective, groups))
        train, train_obj = data.iloc[train_index].reset_index(drop=True),objective.iloc[train_index].reset_index(drop=True)
        eval_data, eval_obj = data.iloc[eval_index].reset_index(drop=True).drop(columns=["case"]), objective.iloc[eval_index].reset_index(drop=True)
        
        def obj(trial):
            return optimal_params(trial, data=train, objective=train_obj)

        study = optuna.create_study(direction='maximize')
        study.optimize(obj, n_trials=25)

        train = train.drop(columns=["case"])
        params = study.best_params
        params["enable_categorical"] = True
        xgbc = XGBClassifier(**params)
        xgbc.fit(X=train,y=train_obj)
        #prediction = xgbc.predict_proba(X=eval_data)[:,1]
        prediction = xgbc.predict(X=eval_data)
        #final_score = average_precision_score(y_true=eval_obj,y_score=prediction)
        report = classification_report(y_true=eval_obj,y_pred=prediction,output_dict=True)
        conf_matrix = confusion_matrix(y_true=eval_obj,y_pred=prediction) #[[TN FP],[FN TP]]
        
        res[prolapses[p]] = Results(report,conf_matrix)
    return res
    
    

# Ejecución del experimento

In [ ]:
res =main_func()
res

[I 2026-02-15 20:04:37,271] A new study created in memory with name: no-name-5721e097-3d2d-4c98-bbb5-0a39daef8973


## Registrar los resultados

In [ ]:
context = "Reporte de resultados con la columna de casos eliminada, usando average_precission_score para la optimización, scale_pos_weight para potenciar aquellos casos en los que hay pocos casos positivos"
generate_html_report(results, "current_experiment.html", context)

# Representación y almacenamiento de los resultados

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt
import io
import base64
import os

In [5]:
def _matrix_to_base64(matrix, title):
    """Genera la imagen de la matriz y la codifica en base64."""
    plt.figure(figsize=(5, 4))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred False', 'Pred True'],
                yticklabels=['Actual False', 'Actual True'])
    plt.title(f'Matriz de Confusión: {title}')
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close()
    return base64.b64encode(buf.getvalue()).decode('utf-8')

In [11]:
def generate_html_report(results, file_name="reporte_final.html", context=""):
    """
    Genera el reporte HTML en results/base con precisión de 5 decimales.
    Permite añadir una línea de contexto bajo el título principal.
    """
 
    # Asegurar extensión .html
    if not file_name.lower().endswith(".html"):
        file_name += ".html"

    # Definir ruta absoluta
    directorio_actual = os.getcwd()
    ruta_destino = os.path.join(directorio_actual, "results", "base")

    # Crear carpetas si no existen
    os.makedirs(ruta_destino, exist_ok=True)

    path_completo = os.path.join(ruta_destino, file_name)

    # Estructura del documento
    html = f"""
    <html>
    <head>
        <meta charset="UTF-8">
        <style>
            body {{ font-family: sans-serif; margin: 30px; background-color: #f4f7f6; color: #333; }}
            h1 {{ text-align: center; color: #2c3e50; margin-bottom: 5px; }}
            .contexto {{ text-align: center; color: #7f8c8d; font-style: italic; margin-bottom: 40px; font-size: 1.1em; }}
            .card {{ background: white; border-radius: 10px; padding: 20px; margin-bottom: 30px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }}
            h2 {{ color: #2980b9; border-bottom: 1px solid #eee; padding-bottom: 10px; }}
            .container {{ display: flex; flex-wrap: wrap; gap: 20px; align-items: center; }}
            .table-wrap {{ flex: 1; min-width: 450px; overflow-x: auto; }}
            table {{ border-collapse: collapse; width: 100%; font-variant-numeric: tabular-nums; font-size: 13px; }}
            th, td {{ border: 1px solid #ddd; padding: 10px; text-align: center; white-space: nowrap; }}
            th {{ background-color: #2980b9; color: white; }}
            tr:nth-child(even) {{ background-color: #f9f9f9; }}
        </style>
    </head>
    <body>
        <h1>Reporte de resultados de Enfermedades</h1>
        <div class="contexto">{context}</div>
    """

    for enfermedad, datos in results.items():
        img_str = _matrix_to_base64(datos['Matrix'], enfermedad)
        
        # DataFrame con 5 decimales
        df_report = pd.DataFrame(datos['Report']).transpose().round(5)
        title = enfermedad.replace('_', ' ').upper()

        html += f"""
        <div class="card">
            <h2>{titulo}</h2>
            <div class="container">
                <div style="flex: 0 0 400px;">
                    <img src="data:image/png;base64,{img_str}" width="100%">
                </div>
                <div class="table-wrap">
                    {df_report.to_html()}
                </div>
            </div>
        </div>
        """

    html += "</body></html>"

    # Escritura del archivo
    with open(path_completo, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Archivo guardado en: {path_completo}")